# External Behavior Model: Offline Training Pipeline

This notebook contains the offline training pipeline for the external behavioral intelligence model. It is designed to train on public transactional datasets like **PaySim** and **IEEE-CIS Fraud Detection**.

**Note for Hackathon Demo**: 
At runtime, the Assurance Engine will solely load the `.joblib` model artifact produced here. The external ML model provides an *advisory signal* only, ensuring deterministic financial invariants remain the absolute source of truth.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we can import the backend adapters
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'backend')) if os.getcwd().endswith('notebooks') else os.path.abspath(os.path.join(os.getcwd(), 'backend')))
from app.learning.adapters.paysim_adapter import PaySimAdapter
from app.learning.adapters.ieee_cis_adapter import IeeeCisAdapter

## 1. Dataset Loading

We load the full datasets. If the full datasets are not present locally (as they are gigabytes in size), the pipeline degrades gracefully to the sample fixtures used for testing.

In [ ]:
REAL_PAYSIM_PATH = '../backend/data/external/paysim/PS_20174392719_1491204439457_log.csv' if os.getcwd().endswith('notebooks') else 'backend/data/external/paysim/PS_20174392719_1491204439457_log.csv'
REAL_IEEE_PATH = '../backend/data/external/ieee_cis/train_transaction.csv' if os.getcwd().endswith('notebooks') else 'backend/data/external/ieee_cis/train_transaction.csv'

FIXTURE_PAYSIM_PATH = '../backend/data/external/fixtures/paysim_sample.csv' if os.getcwd().endswith('notebooks') else 'backend/data/external/fixtures/paysim_sample.csv'
FIXTURE_IEEE_PATH = '../backend/data/external/fixtures/ieee_cis_sample.csv' if os.getcwd().endswith('notebooks') else 'backend/data/external/fixtures/ieee_cis_sample.csv'

paysim_adapter = PaySimAdapter()
ieee_adapter = IeeeCisAdapter()

paysim_path = REAL_PAYSIM_PATH if os.path.exists(REAL_PAYSIM_PATH) else FIXTURE_PAYSIM_PATH
ieee_path = REAL_IEEE_PATH if os.path.exists(REAL_IEEE_PATH) else FIXTURE_IEEE_PATH

print(f"Loading PaySim data from: {paysim_path}")
paysim_records = paysim_adapter.ingest(paysim_path) if os.path.exists(paysim_path) else []

print(f"Loading IEEE-CIS data from: {ieee_path}")
ieee_records = ieee_adapter.ingest(ieee_path) if os.path.exists(ieee_path) else []

all_records = paysim_records + ieee_records
print(f"Total canonical ExternalBehavioralRecords loaded: {len(all_records)}")

## 2. Feature Extraction & Preprocessing

The Adapters have already normalized the features into a canonical representation. We will extract them into a format suitable for scikit-learn.

In [ ]:
X = []
y = []
feature_names = []

if all_records:
    # Standardize feature ordering (as done in adapter to_vector() method)
    feature_names = sorted(all_records[0].behavioral_features.keys())
    
    for record in all_records:
        X.append(record.to_vector())
        y.append(record.external_fraud_label)

X = np.array(X)
y = np.array(y)

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

## 3. Train / Validation Split

In [ ]:
# To demonstrate training, we need a mix of class 0 and 1.
# If using the tiny fixture and it lacks class balance, we mock the labels strictly for the pipeline demo.
if len(set(y)) < 2 and len(y) > 1:
    print("WARNING: Insufficient class balance in fixtures. Mocking targets for demonstration purposes.")
    y[0] = 1 if y[0] == 0 else 0

if len(X) > 5 and len(set(y)) > 1:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
else:
    print("Not enough data to split.")
    X_train, X_val, y_train, y_val = X, X, y, y

## 4. Model Training (Random Forest)

We use a Random Forest Classifier to identify behavioral risk signals.

In [ ]:
print("Training Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)

if len(X_train) > 0 and len(set(y_train)) > 1:
    rf_model.fit(X_train, y_train)
    print("Training complete.")
else:
    print("Insufficient data to train. Please load the real datasets.")

## 5. Evaluation Metrics

Assurance handles strict deterministic bounds. This model provides an *advisory anomaly score*, so we care highly about Recall and ROC-AUC.

In [ ]:
if hasattr(rf_model, 'classes_'):
    y_pred = rf_model.predict(X_val)
    y_prob = rf_model.predict_proba(X_val)[:, 1] if len(rf_model.classes_) > 1 else np.zeros(len(X_val))
    
    print("Classification Report:")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    try:
        roc_auc = roc_auc_score(y_val, y_prob)
        print(f"ROC-AUC Score: {roc_auc:.4f}")
    except ValueError:
        print("ROC-AUC not calculable (single class present in y_val).")
        
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Validation Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

## 6. Feature Importance

In [ ]:
if hasattr(rf_model, 'feature_importances_') and len(feature_names) > 0:
    importances = rf_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(8,4))
    plt.title("Behavioral Feature Importances")
    plt.bar(range(len(importances)), importances[indices], align="center")
    plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Save Model Artifact

The trained model is exported to `.joblib`. The backend Assurance `ExternalBehaviorModel` class strictly loads this file at runtime.

In [ ]:
MODEL_DIR = '../backend/data/models' if os.getcwd().endswith('notebooks') else 'backend/data/models'
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'external_behavior_model.joblib')

if hasattr(rf_model, 'classes_'):
    joblib.dump(rf_model, MODEL_PATH)
    print(f"Model successfully exported to: {MODEL_PATH}")

## 8. Inference Example

Simulating how the backend will evaluate an incoming transaction.

In [ ]:
if hasattr(rf_model, 'classes_') and len(X_val) > 0:
    sample_features = dict(zip(feature_names, X_val[0]))
    print(f"Incoming behavioral features: {sample_features}")
    
    # Convert to vector same as backend does
    sorted_keys = sorted(sample_features.keys())
    vector = [[float(sample_features.get(k, 0.0)) for k in sorted_keys]]
    
    prob = rf_model.predict_proba(vector)[0]
    signal = float(prob[1]) if len(prob) > 1 else 0.0
    
    print(f"\n===> External Behavior Signal: {signal:.4f} (Advisory Only)")